In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *


In [0]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
catalog           = "charles_schwab_retailbrokerage_dev_team_lemma"
bronze_watch      = f"{catalog}.bronze.watchhistory"
silver_watches    = f"{catalog}.silver.watches"


In [0]:
# ─── STEP 1: Read bronze.watchhistory (all batches) ──────────────────────
# B1: 3,000,195 rows  |  B2: 6,897 rows  |  B3: 6,897 rows
# All STRING in Bronze, ALL batches for full rebuild

spark.read.table(bronze_watch).createOrReplaceTempView("v_bronze_watch")
source_count = spark.sql("SELECT COUNT(*) FROM v_bronze_watch").first()[0]



row = spark.sql(f"SELECT _run_id, _batch_id FROM {bronze_watch} ORDER BY _ingest_ts DESC LIMIT 1").first()
carried_run_id = row[0]
carried_batch = row[1]

print(f"bronze.watchhistory rows : {source_count}")
print(f"carried_run_id           : {carried_run_id}")
print(f"carried_batch            : {carried_batch}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_watchhistory', f'Starting processing for WatchHistory (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'RUNNING')

In [0]:
# ─── STEP 2: Normalize & Cast ─────────────────────────────────────────────
# B1 schema (4 cols): W_C_ID, W_S_SYMB, W_DTS, W_ACTION
# B2/B3 schema (6 cols): CDC_FLAG, CDC_DSN added as prefix
#
# Since bronze.customermgmt landing_to_bronze already handles schema merging
# (mergeSchema=true), all rows in bronze have all 6 columns but B1 rows
# have NULL in CDC_FLAG and CDC_DSN — that's fine for our processing.
#
# Type casts required:
#   W_C_ID  → BIGINT
#   W_DTS   → TIMESTAMP
#   W_S_SYMB, W_ACTION remain STRING

df_silver = spark.sql("""
    SELECT
        CAST(W_C_ID    AS BIGINT)     AS W_C_ID,
        W_S_SYMB,
        CAST(W_DTS     AS TIMESTAMP)  AS W_DTS,
        W_ACTION,
        -- Normalize CDC_FLAG: B1 rows have NULL → treat as 'I' (they are all initial loads)
        COALESCE(CDC_FLAG, 'I')       AS CDC_FLAG,
        _batch_id                     AS _batch,
        _run_id,
        _ingest_ts,
        current_timestamp()           AS _load_ts
    FROM v_bronze_watch
""")

df_silver.createOrReplaceTempView("v_watch_typed")
print(f"Rows after type cast: {df_silver.count()}")


In [0]:
# ─── STEP 3: Deduplicate by (W_C_ID, W_S_SYMB, W_DTS) ───────────────────
# If the same watch event was accidentally ingested twice, keep only the latest
# This is idempotent — running it again with same data gives same result

df_watch_dedup = spark.sql("""
    SELECT * EXCEPT (rn, _ingest_ts)
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY W_C_ID, W_S_SYMB, W_DTS
                ORDER BY _ingest_ts DESC       -- latest ingestion wins
            ) AS rn
        FROM v_watch_typed
    )
    WHERE rn = 1
""")

df_watch_dedup.createOrReplaceTempView("v_watch_dedup")
dedup_count = df_watch_dedup.count()
print(f"Rows after dedup: {dedup_count}")


In [0]:
# ─── STEP 4: CREATE OR REPLACE silver.watches (full rebuild) ─────────────
# MD spec: "Silver: CREATE OR REPLACE (full rebuild)"
# Because WatchHistory is append-only and we rebuild from all bronze data each time,
# we simply OVERWRITE the entire silver table.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver;")
df_watch_dedup.write.format("delta").mode("overwrite").saveAsTable(silver_watches)

# ─── PySpark equivalent using SQL (commented for reference) ──────────────
# spark.sql(f"""
#     CREATE OR REPLACE TABLE {silver_watches}
#     USING DELTA
#     AS SELECT * FROM v_watch_dedup
# """)

silver_count = spark.read.table(silver_watches).count()
print(f"silver.watches rows: {silver_count}")

# Breakdown by W_ACTION (ACTV = placed, CNCL = cancelled)
print("W_ACTION breakdown:")
spark.read.table(silver_watches).groupBy("W_ACTION").count().show()


In [0]:

# ─── STEP 5: Operations Audit Logging ────────────────────────────────────
log_pipeline_recon(
    spark=spark, run_id=carried_run_id, batch_id="ALL",
    domain="CUSTOMER", table_name="watches",
    source_layer="bronze", target_layer="silver",
    source_count=source_count, target_count=silver_count
)
log_audit_event(
    spark=spark, run_id=carried_run_id, batch="ALL",
    layer="silver", table_name="watches",
    operation="OVERWRITE", rows_affected=silver_count
)

null_timestamp_count = spark.sql(f"SELECT COUNT(*) FROM {silver_watches} WHERE W_DTS IS NULL").first()[0]
log_dq_result(spark, carried_run_id, "silver.watches", "Null Timestamp Check", null_timestamp_count, silver_count)

log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_watchhistory', 'Successfully completed processing WatchHistory')
